# Variational Autoencoder Applied to MNIST Dataset

Source:<P>

https://kikaben.com/vae-2013/<P>

Adapted:<P>

Antonio Esteves @ UMinho :: Mar 2024

Variational Autoencoders (VAEs) are a powerful tool in machine learning. Unlike traditional Autoencoder models that primarily replicate input data, VAEs can generate new outputs. They achieve this by identifying and learning hidden features from training datasets and using them as a blueprint to generate new data.

Diederik Kingma and Max Welling, from the University of Amsterdam, significantly advanced this field with their Auto-Encoding Variational Bayes paper. This pioneering work introduced AEVB, a novel approach to training generative models. AEVB name symbolizes two crucial functions:

* **AE** for compressing and reconstructing data.
* **VB** for approximating complex data distributions by optimizing towards simpler ones.

Using the AEVB framework, we derive VAEs, generative models capable of creating content like images. While AEVB applies to various data types, this notebook deals with images.

# 1 The Big Question

The paper Auto-Encoding Variational Bayes introduced Variational Auto-Encoder (VAE), starting with a question:

"How can we perform efficient inference and learning in directed probabilistic models, in the presence of continuous latent variables with intractable posterior distributions, and large datasets?", _Auto-Encoding Variational Bayes_.

This question encapsulates the central challenge that VAEs aim to address.

## 1.1 Directed Probabilistic Models: The Big Picture

Directed probabilistic models, commonly known as Bayesian networks, utilize a directed acyclic graph (DAG) to illustrate the relationships and dependencies among the various random variables of a model. Consider the simple DAG below, which depicts the dependencies between random variables $A$, $B$, and $C$:

![](../fig/vae_kikaben_a_b_c.png)

In this graph, the nodes symbolize random variables, while the edges indicate conditional dependencies. An edge from node $A$ to node $B$ means that random variable $B$ depends on random variable $A$. This relationship is **directed** because it follows a specific direction, from $A$ to $B$, and not vice versa. The term **probabilistic** in the context of these models alludes to their foundation in probability theory, with the arrows in the graph representing the conditional dependencies between these variables.

Let us transition from this basic diagram to the more intricate Variational Autoencoder structure.

![](../fig/vae_kikaben_model1.png)

This figure presents two interconnected DAGs: the **encoder** and the **decoder**. The encoder flow, defined by the dotted lines, demonstrates the compression of an input image $x$ into a latent representation $z$ within the feature space. The symbol $\phi$ represents the encoder parameters.

![](../fig/vae_kikaben_x_encoder_z.png)

The decoder flow, indicated by the solid line, illustrates the reconstruction process, transforming the latent representation $z$ into an image resembling the original input $x$. Here, $\theta$ denotes the decoder parameters.

![](../fig/vae_kikaben_z_decoder_x_dash.png)

The encoder and decoder are probabilistic, and their probability distributions exhibit conditional dependencies, as the two flows have a connection via the latent representation $z$.

![](../fig/vae_kikaben_x_encoder_z_decoder_x_dash.png)

The encoder is responsible for modeling the conditional probability distribution $P_\theta(z \mid x)$, as it takes an input image $x$ and produces a latent representation $z$. The decoder is responsible for modeling the conditional/joint probability distribution $P_\theta(x \mid z)$, as it takes latent variables $z$ and reconstructs an image $x'$ close to the input image $x$.

## 1.2 Continuous Latent Variables: Hidden Features

In the context of VAEs, the variable $z$ captures the underlying latent features of the input data. While individual aspects of an image, such as specific colors or shapes, might be directly discernible, the latent features delve deeper, representing higher-level abstractions inferred from the raw data. These abstracted features, often hidden within the pixels but consistent across many images, capture common characteristics across diverse inputs. The encoder distills the input image $x$ into this abstract representation, transcending immediate pixel values to capture the image essence.

The latent space has two key features:

**Compactness**: The latent space of VAEs is typically of lower dimensionality than the original data. This compact representation ensures the VAE focuses on the most salient and generalizable features. By reducing dimensionality, the model abstracts away from granular, pixel-by-pixel details, centering its attention on the image essence.

**Continuity**: The latent variables $z$ are continuous, enabling smooth and coherent transitions within the latent space. This continuity ensures that even small perturbations in the latent variables can be mapped to meaningful variations in the generated output, allowing VAEs to represent a broad spectrum of features. As the decoder leverages these continuous latent variables, it crafts images that resonate with the original style and content without being mere replicas.

Together, these characteristics of the latent space empower VAEs to generate diverse and novel samples while retaining the fundamental attributes of the training data.

Suppose we can determine the distribution of the latent variables and sample from the latent space. In this case, we can use the decoder to generate new images that mirror the style and ambiance of the training dataset without the need for the input image or encoder. Ultimately, this is what we want, use the decoder as an image generator.

## 1.3 Intractable Posterior Distributions: A Tough Challenge

Let us recall the encoder DAG intruced above:

![](../fig/vae_kikaben_x_encoder_z.png)

The encoder task is to capture the essence of the input image $x$ and represent it in the latent space by modeling the conditional probability $P_\theta(z \mid x)$, known as the **posterior** distribution. The posterior represents the probability of our latent variables $z$ given the observed data $x$. However, accurately determining this distribution is rather complex due to the intricate relationships between the data and latent variables. This complexity makes the posterior intractable.

According to Bayes’ theorem, we can compute the posterior distribution $P_\theta(z \mid x)$ as follows:

\begin{align}
P_\theta(z \mid x) = \frac{P_\theta(x \mid z) P(z)}{P_\theta(x)} 
\end{align}

Looking at the right-hand side of the equation, we see that the decoder models the likelihood $P_\theta(x \mid z)$, the probability of observing an image $x$ given the latent variables $z$. The **prior** distribution $P(z)$ captures our beliefs or assumptions about the latent variables $z$, before observing any image $x$. In Bayesian inference, we refer to it as our prior beliefs. While we might typically assume it to be a simple distribution, such as a Gaussian, there a few alternatives to select the prior:

* An **informative prior** reflects known information or beliefs about a parameter.
* A **non-informative** or **flat prior** is used when there is a lack of prior knowledge, assigning equal weight to all parameter values.
* **Conjugate priors** are chosen for mathematical convenience, ensuring the posterior distribution belongs to the same family as the prior.

These priors encapsulate our initial assumptions before any data observation and can influence the results of Bayesian inference.

In the context of VAEs, our choice of prior is often a standard normal distribution and is motivated by computational convenience and the intention of imposing a specific structure to the latent space. While this choice aligns with the Bayesian principle of incorporating prior beliefs, in the case of VAEs, it is more about "what we want it to be" in terms of efficiency and properties rather than strictly about "what we believe it to be". In short, we want the distribution of latent variables to be a standard normal distribution because it makes our model simple and easy to sample from.

Returning to the Bayes’ formula, the denominator $P_\theta(x)$ represents the marginal likelihood or evidence. It makes the joint probability in the denominator a valid probability distribution. To calculate it, we must integrate (or sum) the joint probability of the image $x$ and latent variable $z$, across all the values of $z$. When $z$ is continuous, we can represent the evidence $P_\theta(x)$ as:

\begin{align}
P_\theta(x) = \int P_\theta(x, z) dz = \int P_\theta(x \mid z) P(z) dz
\end{align}

If we could compute $P_\theta(x)$, we would use the posterior distribution $P_\theta(z \mid x)$ to sample the latent features. However, in most cases and especially when dealing with high-dimensional data, computing the previous integral is intractable. If computing $P_\theta(x)$ is intractable, then we can can not obtain the posterior $P_\theta(z \mid x)$.

## 1.4 Efficient Inference: Variational Inference

Given the challenge of the intractable posterior distribution of the latent variables $z$, how do VAEs perform efficient inference and learning? The answer lies in a technique called variational inference. Variational inference (VI) is a method used to approximate complex, often intractable, posterior distributions with simpler, more tractable ones. The core idea lies in two main steps:

* **Choose an approximate distribution**: Select a family of distributions, typically simpler than the true posterior, to act as an approximation. These distributions have parameters that we can adjust to make the approximation better.
* **Optimize the approximation** by minimizing a certain distance measure to the true posterior: Adjust the parameters of the approximate distribution based on observed data to make it as close as possible to the true posterior. The measure of "closeness" is usually the Kullback-Leibler (KL) divergence.

While the true posterior might be complex and intractable across its entire domain, the beauty of VI lies in its locality. Instead of attempting a global approximation that fits the entire distribution, VI focus on regions relevant to the observed data. By focusing on these local regions, VI can leverage simpler distributions to approximate the complex behavior of the true posterior where it matters most. This selective approach is why a seemingly simpler distribution can approximate a more intricate one.

Here is an analogy that might further clarify the locality of variational inference. Imagine we are trying to understand the shape of a complex mountain range with peaks, valleys, and intricate terrains. If we try a global approximation, we attempt to fit a single smooth curve to capture the entire range. That would be challenging, computationally intense, and might miss many details. But we may think of another approach: instead of mapping the whole range, focus on small sections. We fit curves to these local areas, capturing their details accurately. Over time, we aim to approximate the entire range more accurately by piecing together our understanding of many such sections. This strategy of focusing on specific areas or regions, then stitching them together for a broader understanding, mirrors the principle of local approximation in VI.

The next challenge is how can we fine-tune the parameters of our approximate function.

## 1.5 Efficient Learning: Deep Learning

Let us denote our approximate posterior distribution as $Q_\phi(z \mid x)$. Here, $\phi$ represents the parameters that we can adjust to fit $Q_\phi(z\mid x)$ to the true posterior $P_\theta(z \mid x)$. The main challenge is determining how to adjust these parameters efficiently.

We are working with images, than it is a high-dimensional space, and we need an approach that can dynamically adjust $Q_\phi(z\mid x)$ based on our observed data $x$. This makes us think on neural network to parameterize the approximate function. Thus in VAEs, we often use neural networks as an efficient way to parameterize and optimize our approximate distribution. Specifically, given an image $x$, the neural network outputs the parameters of the distribution $Q_\phi(z\mid x)$, such as the mean and variance, from which we can sample the latent variable $z$.

Neural networks can handle high-dimensional data and large datasets using techniques like stochastic gradient descent. By defining a loss function, we can adjust the parameters $\phi$ of our network, efficiently making $Q_\phi(z\mid x)$ a better approximation of $P_\theta(z \mid x)$. To define the loss we will consider the Kullback-Leibler (KL) divergence, which measures how one probability distribution differs from another. As we want to make $Q_\phi(z\mid x)$ closer to the true posterior $P_\theta(z \mid x)$, we want to minimize the KL divergence between the two distributions, which we can include in the loss function as a regularization term.

But if $P_\theta(z \mid x)$ is intractable, we cannot compute the KL divergence between $Q_\phi(z\mid x)$ and $P_\theta(z \mid x)$. However, we can maximize the Evidence Lower Bound (ELBO) derived from the KL divergence. By maximizing the ELBO, we implicitly minimize the KL divergence between the approximate distribution $Q_\phi(z\mid x)$ and the true posterior $P_\theta(z \mid x)$, even though we do not compute the divergence directly.

# 2 The Inner Workings of VAEs

## 2.1 Evidence Lower Bound (ELBO)

In Variational Autoencoders, the Evidence Lower Bound (ELBO) plays a pivotal role. It is a surrogate objective function to optimize our model even when the actual posterior distribution is intractable. Let us derive the ELBO from the KL divergence.

The Kullback-Leibler (KL) divergence between the approximate posterior $Q_\phi(z\mid x)$ and the true posterior $P_\theta(z \mid x)$ is given by:

\begin{align}
D_{KL}(Q_\phi(z \mid x) \parallel P_\theta(z \mid x)) = \mathbb{E}_{Q_\phi(z \mid x)}[\log Q_\phi(z \mid x) - \log P_\theta(z \mid x)]
\end{align}

The above KL divergence formula measures the divergence between our approximate distribution $Q_\phi(z\mid x)$ and the true posterior $P_\theta(z \mid x)$. The expectation is computed over the approximate distribution $Q_\phi(z\mid x)$, that we control and know. So, we calculate the KL divergence based on $Q_\phi(z\mid x)$, but the formula still has the intractable $P_\theta(z \mid x)$. Let see how we can circumvent this problem.

\begin{align}
P_\theta(z \mid x) = \frac{P_\theta(x \mid z) P(z)}{P_\theta(x)}
\end{align}

Substituting this into the KL divergence we get:

\begin{align}
\small{
\begin{aligned}
D_{KL}(Q_\phi(z \mid x)  \parallel  P_\theta(z \mid x)) &= \mathbb{E}_{Q_\phi(z \mid x)}\biggl[\log Q_\phi(z \mid x) - \log P_\theta(z \mid x)\biggr] \\\\
&= \mathbb{E}_{Q_\phi(z \mid x)}\left[\log Q_\phi(z \mid x) - \log \frac{P_\theta(x \mid z) P(z)}{P_\theta(x)}\right] \\\\
&= \mathbb{E}_{Q_\phi(z \mid x)}\biggl[\log Q_\phi(z \mid x) - \log P_\theta(x \mid z) - \log P(z) + \log P_\theta(x)\biggr] \\\\
&= D_{KL}(Q_\phi(z \mid x)  \parallel  P(z)) - \mathbb{E}_{Q_\phi(z \mid x)}[\log P_\theta(x \mid z)] + \log P_\theta(x)
\end{aligned}
}
\end{align}

Rearranging the terms:

\begin{align}
\small{
\mathbb{E}_{Q_\phi(z \mid x)}[\log P_\theta(x \mid z)] - D_{KL}(Q_\phi(z \mid x)  \parallel  P(z)) = \log P_\theta(x) - D_{KL}(Q_\phi(z \mid x)  \parallel  P_\theta(z \mid x))
}
\end{align}

The left-hand side is what we refer to as the Evidence Lower Bound (ELBO). Therefore, we can define the ELBO as:

\begin{align}
\text{ELBO} = \mathbb{E}_{Q_\phi(z \mid x)}[\log P_\theta(x \mid z)] - D_{KL}(Q_\phi(z \mid x)  \parallel  P(z))
\end{align}

In VAEs, we use the **decoder** to model the generative process represented by the distribution $P_\theta(x \mid z)$. $P_\theta(x \mid z)$ represents the probability distribution of observing the image $x$ given the latent variables $z$, and it is modeled by the decoder. $P(z)$ represents our chosen prior distribution for the latent variables.


The ELBO expression does not include the intractable true posterior $P_\theta(z \mid x)$, and we can use it for two purposes:

* **Maximizing the likelihood of the data**: The term $\mathbb{E}_{Q_\phi(z \mid x)}[\log P_\theta(x \mid z)]$ represents the expected log-likelihood of the image $x$ given the latent variables $z$. By maximizing this term, we aim to ensure that the data reconstructed by the decoder is as close as possible to the original image data $x$.

* **Regularizing the latent space**: The term $D_{KL}(Q_\phi(z \mid x) \parallel P(z))$ acts as a regularizer. It ensures that the distribution of the latent variables $z$, as modeled by the encoder, does not deviate too much from the prior distribution that we selected. This term encourages the latent space to maintain a desired structure, allowing us to sample $z$ to generate new images.

By maximizing the ELBO, we achieve these two objectives: we ensure that our VAE reconstructs the data accurately while maintaining a structured latent space.

We can also define ELBO in a different way:

\begin{align}
\text{ELBO} = \log P_\theta(x) - D_{KL}(Q_\phi(z \mid x)  \parallel  P_\theta(z \mid x))
\end{align}

While this representation of the ELBO may look different from the previously discussed one, it is an equivalent definition using different terms based on the earlier derivation of the ELBO.

This version of ELBO includes the intractable true posterior $P_\theta(z \mid x)$. However, it tells us that maximizing ELBO means maximizing the evidence $P_\theta(x)$ and minimizing the KL divergence between the approximate posterior $Q_\phi(z\mid x)$ and the true posterior $P_\theta(z \mid x)$, which is why maximizing ELBO ensures our approximate posterior becomes closer to the intractable true posterior $P_\theta(z \mid x)$, without ever calculating it. Moreover, the ELBO provides a lower bound on the log evidence as any KL divergence is non-negative:

\begin{align}
\log P_\theta(x) \ge \text{ELBO}
\end{align}

The ELBO becomes equal to the logarithm of the evidence only when the $Q_\phi(z\mid x)$ and $P_\theta(z \mid x)$ are the same.

Having derived the ELBO, we now face the challenge of optimizing it. Let us discuss that in the following sections.

## 2.2 Encoder: From Images to Distributions in Latent Space

Unlike traditional autoencoders, which directly map an input to a point in the latent space, VAEs map an input to a distribution in the latent space. This probabilistic approach recognizes the inherent uncertainty when representing complex data, like images, in a lower-dimensional latent space.

The encoder in a VAE, often implemented as a convolutional neural network (CNN), processes an input image and estimates the distribution of the latent variables that correspond to that image. More specifically, for each input image $x$, the encoder predicts the mean and variance of the latent variables $z$, locally approximating the posterior distribution for that image.

![](../fig/vae_kikaben_x_encoder_mu_sigma.png)

So, the encoder processes an input image $x$ and estimates the distribution parameters of the latent variables $z$. These estimates are constrained since the encoder output is regulated by the KL divergence term of the ELBO. The $D_{KL}$ term functions as a regularizer that quantifies the divergence between the distribution $Q_\phi(z \mid x)$ estimated by the encoder and our selected prior distribution $P(z)$. This term ensures that the latent space is well-structured and not scattered randomly.

We assume that each latent variable $z \sim P(z)$ follows a standard normal distribution, as this choice simplifies the KL divergence term in the ELBO and often results in a well-behaved latent space. It is worth noting that this choice is not a restriction imposed by the VAE framework, it is a design decision. The prior could be another distribution, depending on the actual problem to solve. To keep the discussion simpler, let us consider a 1D latent space.

\begin{align}
P(z) = \mathcal{N}(z; 0, 1)
\end{align}

where $\mathcal{N}$ represents the Normal (or Gaussian) distribution.

For $Q_\phi(z \mid x)$, we have:

\begin{align}
Q_\phi(z \mid x) = \mathcal{N}(z; \mu_{\phi,x}, \sigma^2_{\phi,x})
\end{align}

Here, $\mu_{\phi,x}$ and $\sigma^2_{\phi,x}$ indicate that we get a different Gaussian distribution for each input $x$. In other words, based on $x$, the encoder predicts a mean $\mu$ and variance $\sigma^2$ for latent variable $z$. In this way, the encoder provides the local approximation.

Given large datasets, the objective is to minimize the KL divergence between $Q_\phi(z \mid x)$ and $P(z)$ across many images. It ensures that, in the aggregate, the distributions $Q_\phi(z \mid x)$ across various inputs will converge to approximate the prior $P(z)$, which we design as the standard normal distribution.

In the training phase, the encoder task is to estimate the parameters of $Q_\phi(z \mid x)$ for each image. The KL divergence then serves as a regularization term in the loss function, guiding the encoder toward our desired prior distribution.

As we selected the approximate posterior $Q_\phi(z \mid x)$ and the prior $P(z)$ as Gaussian distributions, we can derive the KL divergence between these distributions by the following derivation.

\begin{align}
\small{
\begin{aligned}
D_{KL}(Q_\phi(z \mid x) \ \mid  P(z)) &= \int Q_\phi(z \mid x) \log \left( \frac{Q_\phi(z \mid x)}{P(z)} \right) dz \\
&= \int Q_\phi(z \mid x) \biggl[\ \log Q_\phi(z \mid x) - \log P(z) \ \biggr] dz \\
&= \int Q_\phi(z \mid x) \biggl[\ \log \frac{1}{\sqrt{2\pi\sigma^2_{\phi,x}}} \exp \left( - \frac{(z - \mu_{\phi,x})^2}{2\sigma^2_{\phi,x}} \right) 
 - \log \frac{1}{\sqrt{2\pi}} \exp \left( - \frac{z^2}{2} \right) \ \biggr] dz \\
&= \int Q_\phi(z \mid x) \biggl[\ -\frac{1}{2} \log (2\pi\sigma^2_{\phi,x}) - \frac{(z - \mu_{\phi,x})^2}{2\sigma^2_{\phi,x}} - \left( -\frac{1}{2} \log 2\pi - \frac{z^2}{2} \right) \ \biggr] dz \\
&= \frac{1}{2} \int Q_\phi(z \mid x) \biggl[ -\log \sigma^2_{\phi,x} - \frac{(z - \mu_{\phi,x})^2}{\sigma^2_{\phi,x}} + z^2 \biggr] dz \\
&= \frac{1}{2} \biggl( -\log \sigma^2_{\phi,x} - 1 + \mu^2_{\phi,x} + \sigma^2_{\phi,x} \biggr)
\end{aligned}
}
\end{align}

In the last step, we used the relationship $\mathbb{E}[z^2] = \mu^2 + \sigma^2$.

For a VAE with $J$ independent latent variables, we sum this value over all variables.

\begin{align}
D_{KL}(Q_\phi(z \mid x) \ \mid  P(z)) = \frac{1}{2} \sum_{j=1}^{J} \biggl( -\log \sigma^2_{\phi_j,x} - 1 + \mu^2_{\phi_j,x} + \sigma^2_{\phi_j,x} \biggr)
\end{align} 

This KL divergence measures how much the encoder estimate deviate from the standard normal prior. Minimizing this term during the training encourages the encoder estimated distribution to closely align with the standard normal distribution, facilitating a structured latent space. By estimating the latent distribution for each image across large datasets, the VAE aligns its representations with the designed latent structure.

More intuitively, we use the KL divergence to force the distribution of the latent variables to be a standard normal so that we can sample latent variables from that standard normal distribution. In this setup, the prior distribution is more a a desired shape or structure for our latent space than an initial guess for the distribution of $z$.

VAEs employ a unique strategy in their latent space. Instead of learning a fixed representation for each image, they understand a range of possible representations by sampling different points around the mean. This sampling process is fundamental to the VAE generative capabilities.

During the encoding phase, the VAE estimates the latent variable distribution for an image. However, the broader goal is not just to represent existing images. We want to use this latent space to generate new ones. If the VAE only relied on the mean value, it might limit the diversity of the latent space and hinder the generation of varied images. Sampling from the estimated latent distributions ensures a well-populated and continuous latent space, reinforcing the VAE strength as a generative model.

However, there is a problem. The sampling operation is inherently non-differentiable, which poses a challenge for gradient backpropagation. However, the VAE authors proposed the reparameterization trick to allow the gradients flow through the latent variables sampling.

## 2.3 The Reparameterization Trick

In our discussion so far, we have looked at the VAE through the lens of a Directed Acyclic Graph (DAG) that captures the probabilistic dependencies between variables. This perspective is crucial for understanding the generative process and the relationships between the encoder, latent variables, and decoder.

![](../fig/vae_kikaben_x_encoder_z_decoder_x_dash.png)

However, when training the VAE, we must shift our viewpoint slightly.

Training a VAE, like any deep learning model, involves optimizing a loss function using gradient-based methods. That requires us to compute gradients of the loss with respect to the model parameters. In this context, we should consider the VAE as a computational graph where nodes represent operations and edges represent the flows of data and gradients. So, we need to think about both ways, in feed-forward and back-propagation steps.

As mentioned earlier, a challenge arises when we sample latent variables. Sampling is a stochastic operation and is inherently non-differentiable. That means that we cannot directly propagate the gradients through the sampling step, which poses a problem to backpropagation, the primary algorithm used to train deep neural networks.

The reparameterization trick is a clever workaround that allows us to bypass the non-differentiability of the sampling step. Instead of sampling from the distribution predicted by the encoder, we sample from a standard normal distribution and then shift and scale the sample using the mean and variance predicted by the encoder.

\begin{align}
z = \mu + \sigma \odot \epsilon
\end{align}

Here, $\odot$ denotes element-wise multiplication. This reparameterization allows us to separate the stochasticity from the parameters we want to optimize. The randomness is now in $\epsilon$, which does not depend on $\mu$ or $\sigma$, allowing gradients to flow through $\mu$ and $\sigma$ during backpropagation.

In summary, the reparameterization trick transforms the optimization problem into one in which the randomness is external to the computational graph, enabling gradient-based optimization methods to work. It is a vital aspect of VAEs, allowing them to learn efficiently using standard deep learning frameworks and optimization techniques.

This component of the VAE architecture takes the sampled latent variables and reconstructs the input data, playing a vital role in the VAE’s generative capabilities.

## 2.4 Decoder: Reconstructing Images from Latent Representations

The decoder in a VAE is responsible for translating the latent variables back into the original data space. In the context of images, this means taking the sampled latent variables and producing an image $x'$ that closely resembles the original input $x$.

![](../fig/vae_kikaben_z_decoder_x_dash.png)

At its core, the decoder is a neural network designed to do the opposite of what the encoder does. In more straightforward terms, it transforms the condensed latent vector into a complete image.

The process typically begins with a fully connected layer that takes the latent vector $z$ as input and produces a tensor of suitable shape. This tensor then goes through a set of upsampling layers. These upsampling layers may consist of transposed convolution layers that progressively enlarge the tensor dimension until they match the desired image size.

In essence, the decoder aims to create an image $x'$ that closely resembles the original image $x$. To achieve this, it focuses on maximizing the likelihood $P_\theta(x \mid z)$ of the observed data $x$ given the latent variables $z$. The greater this likelihood, the more proficient the decoder becomes at reconstructing the original data from the latent space.

Considering that $P_\theta(x \mid z)$ follows a Gaussian distribution with mean $x'$, which is the decoder output, and a fixed variance $\sigma^2$, we can represent the likelihood of the entire image as the product of the individual likelihood of each pixel.

\begin{align}
P_\theta(x \mid z) = \prod_{i=1}^S P_\theta(x_i \mid z)
\end{align}

where $S$ is the number of pixels in the image.

The Gaussian likelihood represents the likelihood of the original data $x$, given the average reconstruction $x'$ and the assumed constant variance $\sigma^2$. This variance reflects the inherent uncertainty on data and noise in the reconstruction process.

By taking the logarithm of both sides of the previous equation, we obtain:

\begin{align}
\log P_\theta(x \mid z) = \sum_{i=1}^S \log P_\theta(x_i \mid z)
\end{align}

We can expand the log-likelihood using the assumption that each pixel follows a Normal distribution $\mathcal{N}(x',\sigma^2)$.

\begin{align}
\log P_\theta(x \mid z) =  \sum_{i=1}^S \left[ -\frac{1}{2\sigma^2} (x_i - x'_i)^2 + \text{const} \right] 
\end{align}

\begin{align}
\log P_\theta(x \mid z) = -\frac{1}{2\sigma^2} \sum_{i=1}^S (x_i - x'_i)^2 + S \times \text{constant}
\end{align}

When we maximize this log-likelihood with respect to $x'$ or, equivalently, minimize the negative log-likelihood, the resulting optimization objective is directly proportional to the squared difference between $x$ and $x'$. Given that $\sigma^2$ is fixed, this scaling factor does not alter the optimization direction. As $S \times constant$ includes terms that do not depend on $x$ and $x'$, we can ignore them in the optimization process.

So, the fundamental in the log-likelihood is the squared difference $(x_i - x'_i)^2$ for every pixel. When we add up these squared differences for all pixels, we obtain the sum of squared errors (SSE) between the original and reconstructed images.

\begin{align}
\text{SSE}(x, x') = \sum\limits_{i=1}^S (x_i - x'_i)^2
\end{align}

where $S$ is the number of pixels in the image, and $x_i$ and $x'_i$ are the pixel values at the $i$-th position in the original and reconstructed images, respectively. This SSE quantifies the total squared differences between corresponding pixels in the two images, measuring the reconstruction quality. A lower SSE indicates that the reconstructed image $x'$ is closer to the original image $x$.

Thus, the SSE functions as a reconstruction loss in the VAE that emerged from our Gaussian likelihood assumption.

During the training process with image data, the last layer of the decoder commonly employs a `sigmoid` activation function to ensure that the output pixel values lie in the range [0, 1], matching the normalized pixel values of the original images. If the images are normalized in the range [-1, 1], we can use `tanh` instead. Irrespective of the activation function, the reconstructed image $x'$ is then compared to the original input $x$ to compute the reconstruction loss, guiding the training process to improve the decoder.

The reconstruction loss is also related to the first term in the ELBO.

\begin{align}
\text{ELBO} = \mathbb{E}_{Q_\phi(z \mid x)}[\log P_\theta(x \mid z)] - D_{KL}(Q_\phi(z \mid x)  \parallel  P(z))
\end{align}

The term $\mathbb{E}_{Q_\phi(z \mid x)}[\log P_\theta(x \mid z)]$ represents the expected log-likelihood of the image $x$ given the latent variables $z$, which follow the $Q_\phi(z \mid x)$ distribution. This term measures how well the decoder reconstructs the original data from the latent representation. We aim to maximize this term during training, that is, we want to maximize $-\frac{1}{2\sigma^2} \text{SSE}(x, x')$. This is equivalent to minimize the SSE loss.

We have already discussed the second term, $D_{KL}(Q_\phi(z \mid x)  \parallel  P(z))$, which measures how well the encoder approximates the prior distribution of the latent variables $z$.

# 3 Training a Simple VAE

Now, we will implement in PyTorch a simple version of the encoder and decoder, tailored for the MNIST dataset. MNIST consists of grayscale images of digits with size 28 x 28. We will start with the encoder and decoder classes. 

![](../fig/vae_kikaben_model1.png)

## 3.1 Encoder

The encoder primary role is to capture the essential characteristics of the input data and compress it into a lower-dimensional latent space. Given an image, the encoder outputs two vectors: a mean and a log variance. These vectors define a Gaussian distribution in the latent space from which we can sample the latent variables.

Our encoder includes two blocks, each one composed of a convolution, a ReLU activation function and a MaxPooling layers. These layers help in extracting hierarchical features from the input images. As we move through these layers, the spatial dimension of the feature maps reduce due to the MaxPooling, while the number of channels increases, capturing more features.

After the two mentioned blocks, the feature maps are flattened, passed through a fully connected layer, and finally produce the mean and log-variance vectors. We use log-variance instead of the variance because it is more numerically stable and is not restricted to be positive.

In [ ]:
import torch
import torch.nn            as     nn
import torch.nn.functional as     F
import torch.optim         as     optim
from   torchvision         import datasets, transforms
from   torch.utils.data    import DataLoader

'''
Encoder class, that generates a mean and a log-variance vectors 
that parameterize the inference model q_phi(z|x).
'''
class Encoder(nn.Module):

    def __init__(self, latent_dim: int):
        super().__init__()
        
        # Feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
        )
        
        # Estimate the mean and log-variance
        self.fc1        = nn.Linear(in_features=64*7*7, out_features=400)  # 7x7 feature maps
        self.fc2_mean   = nn.Linear(in_features=400, out_features=latent_dim)
        self.fc2_logvar = nn.Linear(in_features=400, out_features=latent_dim)
    
    def forward(self, x: torch.Tensor) -> (torch.Tensor, torch.Tensor):
        # Feature extraction
        x      = self.feature_extractor(x)

        # Estimate mean and log-variance
        x      = F.relu(self.fc1(x))
        mean   = self.fc2_mean(x)
        logvar = self.fc2_logvar(x)
        return mean, logvar


## 3.2 Decoder

The decoder corresponds to the generative model $P_\theta(x \mid z)$. Given $z=(mean,logvar)$, sampled or directly provided, the decoder tries to reconstruct the original data.

Before upsampling, the decoder has a fully connected layer that takes the latent vector (means,log-variances) as input and expands it into a tensor that matches the dimensions needed for the transposed convolutional layers. This tensor serves as the starting point for the upsampling process.

The decoder includes two transposed convolution layers to perform the upsampling. These layers work inverse to the convolutional layers, gradually increasing the spatial dimensions while reducing the number of channels. The second transposed convolution layer uses a sigmoid activation function to ensure that the pixel values of the reconstructed image are in the range [0, 1], matching the normalized pixel values of the input.

Through these operations, the decoder learns to map samples in the latent space back to a valid image, effectively learning the inverse transformation of the encoder.

In [ ]:
'''
Decoder class, that given a sample (vector) for 'z' generates an image x'.
It parameterize the generative model p_theta(x|z).
'''
class Decoder(nn.Module):

    def __init__(self, latent_dim: int):
        super().__init__()
        
        # Transform latent variables to a suitable shape for upsampling
        self.fc = nn.Sequential(
            nn.Linear(in_features=latent_dim, out_features=64*7*7),
            nn.ReLU()
        )
        
        # Upsampling with transposed convolutions
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=1,  kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()  # Ensuring output is in range [0,1]
        )
    
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # Transform latent variables to a suitable shape
        z = self.fc(z)

        # Reshape z to (batch_size, 64, 7, 7)
        z = z.view(z.size(0), 64, 7, 7)

        # Upsampling for reconstruction
        x_reconst = self.decoder(z)
        return x_reconst


## 3.3 VAE

We can combine the Encoder and Decoder classes to build a VAE class for simultaneously training both the Encoder and Decoder.

The `forward` method first encodes the input image into the latent space, samples from this space using the reparameterization trick, and then decodes the sample back into the data space.

In [ ]:
'''
VAE class than implements the encoder, reparameterization and decoder.
Given an image it returns the reconstructed image, the mean and log-variance 
of the latent variables.
'''
class VAE(nn.Module):

    def __init__(self, latent_dim: int):
        super().__init__()
        
        # Instantiate the Encoder
        self.encoder = Encoder(latent_dim)
        # Instantiate the Decoder
        self.decoder = Decoder(latent_dim)
        
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """
        Reparameterization trick to sample from the latent space.
        """
        std = torch.exp(0.5 * logvar) # get variance from log-variance
        eps = torch.randn_like(std)   # sample from Normal(0,1)
        return mu + std * eps         # z = mu + std * eps

    def forward(self, x: torch.Tensor) -> tuple:
        # Pass the input image through the encoder
        mu, logvar = self.encoder(x)
        
        # Apply the reparameterization trick
        z = self.reparameterize(mu, logvar)
        
        # Pass the latent vector through the decoder
        x_reconstructed = self.decoder(z)
        
        return x_reconstructed, mu, logvar

## 3.4 The Loss Function

During training, we will use the reconstruction loss (the difference between the input `x` and output `x_reconstructed`) and the KL divergence (calculated from `mu` and `log-variance`) to compute the VAE loss.

The reconstruction term of the loss measures how well the decoder has reconstructed the original input. Using the `mse_loss` function with `reduction=‘sum’` calculates the sum of the squared differences between the original and reconstructed images.
The KL divergence term of the loss acts as a regularization term, ensuring that the latent space conforms to a standard normal distribution, aiding in generating new samples.
The model goal during training is to minimize this combined loss, simultaneously improving its reconstruction ability and shaping the learned latent space into the standard normal distributions so that we can sample from it to generate new images.

In [ ]:
def loss_function(reconst_x, x, mu, logvar):
    """
    Compute the VAE loss.
    This loss is the negative of the ELBO formulated above.
    Therefore, minimizing -ELBO is equivalent to maximize the ELBO.
    """

    # Reconstruction loss (summed over all dimensions)
    reconst_loss = F.mse_loss(reconst_x, x, reduction='sum')

    # KL divergence loss (regularization term)
    KL_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Average per image
    batch_size = x.size(0)
    return (reconst_loss + KL_loss)/batch_size # -ELBO

## 3.5 Training Loop of VAE

Below is the main function that contains the training loop of the VAE.

The number of latent dimensions is set to 2 when the VAE is initialized. This choice is not arbitrary; it illustrates the VAE ability to compress data using just two dimensions. It demonstrates the model efficiency and allows us to visualize the images generated in a 2D space.

In [ ]:
from tqdm import tqdm

def main():

    LATENT_DIM = 2
    
    # Set device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using {device} device')
    device = torch.device(device)

    # Load the data
    transform     = transforms.ToTensor()
    
    train_dataset = datasets.MNIST(
        root      = '../data', 
        train     = True,
        transform = transform,
        download  = True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size     = 32,
        shuffle        = True
    )

    # Instantiate the VAE
    model = VAE(latent_dim=LATENT_DIM).to(device)

    # put the model in training mode
    model.train()

    # Select the optimizer
    optimizer = optim.AdamW(model.parameters(), lr=1.0e-3)

    # Train during multiple epochs
    for epoch in tqdm(range(10)):
        
        train_loss = 0

        # Training loop
        for batch_idx, (data, _) in enumerate(train_loader):
            # We only use images not labels
            data = data.to(device)
            
            # Forward pass
            recon_batch, mu, logvar = model(data)
            
            # Backward pass
            optimizer.zero_grad()
            loss = loss_function(recon_batch, data, mu, logvar)        
            loss.backward()
            optimizer.step()

            # Accumulate the loss for logging
            train_loss += loss.item()

            # Print progress information
            if batch_idx % 100 == 0:
                print(f'Train Epoch: {epoch} [{batch_idx * len(data) :5d}/{len(train_loader.dataset) :5d} \
                ({100. * batch_idx / len(train_loader) :2.0f}%)] Loss: {loss.item() / len(data) :8.4f}')

        average_loss = train_loss / len(train_loader.dataset)
        print(f'Epoch: {epoch} Average loss: {average_loss :.4f}')

    # Save the model
    model_path = 'models/vae_model_kikaben.pth'
    torch.save(model.state_dict(), model_path)


In [ ]:
main()

## 3.6 Generating Random New Images

After the training, we can generate new images. The below code generates random images using the trained VAE model. It loads the trained model and gets 49 samples from the standard normal distributions to generate 49 new images.

In [ ]:
import torch
from   torchvision.utils import make_grid
import matplotlib.pyplot as     plt

LATENT_DIM  = 2
num_samples = 49

# 1. Create a new VAE instance and load the saved weights

model2 = VAE(LATENT_DIM)
model2.load_state_dict(torch.load('models/vae_model_kikaben.pth'))

# 2. Draw 'num_samples' samples (each with size LATENT_DIM) 
#    from the latent space (the standard normal)

z  = torch.randn(num_samples, LATENT_DIM)

# 3. Generate 'num_samples' images

model2.eval()
with torch.inference_mode():
    images = model2.decoder(z)

# 4. Visualize the generated images in a grid

grid = make_grid(images, nrow=7, padding=1, pad_value=1)
grid = grid.permute(1, 2, 0)

plt.imshow(grid)
plt.axis('off')
plt.title('Randomly generated images')
plt.show()

Below is an example of possible output from the generation step:

![](../fig/vae_kikaben_new_images.png)

Given only two dimensions in the latent space, the VAE can generate MNIST-like images. Although some of the images are unclear, it clearly shows the ability of the VAE to generate new random images by sampling from the compressed latent space.

## 3.7 Exploring the 2D Latent Space

Now, let us explore the latent space. We step through a 2D grid of values in the latent space and observe how the generated image changes.

In [ ]:
import torch
from   torchvision.utils import make_grid
import matplotlib.pyplot as     plt

LATENT_DIM = 2
steps      = 20

# 1. Create a new VAE instance and load the saved weights

model3 = VAE(LATENT_DIM)
model3.load_state_dict(torch.load('models/vae_model_kikaben.pth'))

# 2. Generate a 2D grid of values (z1,z2) in the latent space
#    where z1 and z2 vary in the range [-2,2] with a step of (2-(-2))/20

latent_values = torch.linspace(-2.0, 2.0, steps)
grid_z = torch.tensor([[z1, z2] for z1 in latent_values for z2 in latent_values])

# 3. Generate images using as input each pair (z1,z2) defined by the grid

model3.eval()
with torch.inference_mode():
    images = model3.decoder(grid_z)


# 4. Visualize the generated images in a 20x20 grid

grid = make_grid(images, nrow=steps, padding=1, pad_value=1)
grid = grid.permute(1, 2, 0)

plt.figure(figsize=(12,12))
plt.imshow(grid)
plt.axis('off')
plt.title('2D latent space exploration')
plt.show()

Below is an example of the output from the latent space exploration.

![](../fig/vae_kikaben_latent.png)

The grid visually demonstrates how adjusting values within the 2D latent space leads to smooth transformations in the generated images. As you move across the grid, you can observe how small changes in the latent values create gradual variations in the images. This continuous relationship between the latent space and the generated images is a powerful feature of VAEs.

Remember that using values far from the mean of the latent distribution (e.g., large positive or negative values) might lead to less clear reconstructions.

While the latent space follows a standard normal distribution and is technically unbounded, the model primarily learns from the range of values that are frequent under this distribution, concentrated around the mean. Values far from the mean might not be well-represented in the model training, leading to less accurate reconstructions.

Knowing the effective range of latent variables is crucial when employing VAEs as practical image-generation tool, such as image augmentation, to control the quality of the generated images and fully leverage the model capabilities.